# Notebook 01: Data audit

**Project:** Perishable Inventory Optimizer (GIS + supply chain)
**Milestone:** 1, Data audit and dataset decision

## What we are trying to accomplish
Check that the Kaggle "Store Sales - Time Series Forecasting" data is complete and trustworthy before we build anything on it: which files we have, how big they are, what date range they cover, which product families exist, and where values are missing.

## What we got accomplished
Steps 1 to 4 are run and explained, the nine-family perishable shortlist is agreed, and the conclusion at the bottom is written.

## What we have left to do
Commit this notebook, then start Milestone 2 (GIS foundation).

**How each step works:** we run the code, look at the result, and fill in the explanation cells below it together: what we are looking for, how to read it, what it means, and what is next and why.

## Step 1: Which files do we have?

### What we are looking for
Confirm the downloaded files are in `data/raw/` and note their sizes. A missing or unusually small file would stop us before we waste time.

In [17]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RAW = Path("../data/raw")
files = sorted(RAW.rglob("*.csv"))  # rglob also searches subfolders (Kaggle unzips into one)

if not files:
    print(f"No CSV files found in {RAW.resolve()}.")
    print("Download the Store Sales files first (see data/raw/README.md).")

for f in files:
    print(f"{f.name:25s} {f.stat().st_size / 1_000_000:8.2f} MB")

holidays_events.csv           0.02 MB
oil.csv                       0.02 MB
sample_submission.csv         0.34 MB
stores.csv                    0.00 MB
test.csv                      1.02 MB
train.csv                   121.80 MB
transactions.csv              1.55 MB


### How to read it
Each line is one file and its size in megabytes. `train.csv` is by far the largest (about 121.8 MB); `transactions.csv` is next (about 1.55 MB); the rest are tiny. (Your folder may also list `test.csv` and `sample_submission.csv`, which are Kaggle competition files we do not need.)

### What it means
The size pattern is what we expect. `train.csv` holds the daily sales, millions of rows, so it is large. The small files are lookup tables: store details, holidays, oil prices. Nothing is missing or suspiciously small, so the download is complete.

### What's next and why
Step 2 opens each file to see what its columns are. We need to know which file holds sales, which holds store locations, and whether anything needs converting, before we can plan the geography and the forecast.

## Step 2: What is in each file?

### What we are looking for
For every file: how many rows and columns, what each column is called, and what type of data it holds. This tells us which files hold sales, which hold store locations, and whether any column needs converting (dates, for example).

In [18]:
tables = {f.stem: pd.read_csv(f) for f in files}

for name, df in tables.items():
    print(f"\n=== {name}: {df.shape[0]:,} rows x {df.shape[1]} columns ===")
    print(df.dtypes.to_string())
    display(df.head(3))


=== holidays_events: 350 rows x 6 columns ===
date            str
type            str
locale          str
locale_name     str
description     str
transferred    bool


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False



=== oil: 1,218 rows x 2 columns ===
date              str
dcoilwtico    float64


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97



=== sample_submission: 28,512 rows x 2 columns ===
id         int64
sales    float64


...

,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2



=== train: 3,000,888 rows x 6 columns ===
id               int64
date               str
store_nbr        int64
family             str
sales          float64
onpromotion      int64


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0



=== transactions: 83,488 rows x 3 columns ===
date              str
store_nbr       int64
transactions    int64


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358


### How to read it
Each block is one file: its name, rows x columns, the type of each column, and the first three rows. Types show how pandas reads a column: `int64` is a whole number, `float64` a decimal, `str` text, `bool` true/false.

Summary of what we found:
- **train**: 3,000,888 rows. One row per store x product family x day: `date`, `store_nbr`, `family`, `sales`, `onpromotion`.
- **stores**: 54 rows. One per store: `city`, `state`, `type`, `cluster`.
- **transactions**: 83,488 rows. Customer transaction count per store per day.
- **oil**: 1,218 rows. Daily oil price (`dcoilwtico`).
- **holidays_events**: 350 rows. Holidays and events, local, regional or national.

### What it means
Sales are recorded per product *family* (for example PRODUCE), not per individual item, and units differ by family (some are counted by weight). That is fine for a first version: we forecast demand for one family at a time.

The only location information is a city and state for each of the 54 stores. There are no coordinates, no warehouses and no costs. That is the main limit of this dataset, and it shapes the GIS plan: we add the geography ourselves (city coordinates, boundaries, assumed warehouse sites) and label every assumption openly.

The `date` columns are read as text, so we convert them to real dates before using them.

### What's next and why
Step 3 checks the dates and product families in the sales file. A forecast needs a long, unbroken history, and we need to confirm the perishable families we want are actually there.

## Step 3: What dates and product families are covered?

### What we are looking for
The first and last date in the sales file, whether any calendar days are missing, and the list of product families. We need to confirm that perishable families (such as produce, dairy and meats) are present, and that the history is long enough to forecast from.

In [19]:
train = tables["train"].copy()
train["date"] = pd.to_datetime(train["date"])

first, last = train["date"].min(), train["date"].max()
calendar_days = (last - first).days + 1
distinct_days = train["date"].nunique()

print(f"Date range: {first.date()} to {last.date()}")
print(f"Calendar days in range: {calendar_days:,}")
print(f"Distinct dates present: {distinct_days:,}")
print(f"Missing calendar days: {calendar_days - distinct_days:,}")

families = sorted(train["family"].unique())
print(f"\nProduct families: {len(families)}")
for fam in families:
    print(" -", fam)

Date range: 2013-01-01 to 2017-08-15
Calendar days in range: 1,688
Distinct dates present: 1,684
Missing calendar days: 4

Product families: 33
 - AUTOMOTIVE
 - BABY CARE
 - BEAUTY
 - BEVERAGES
 - BOOKS
 - BREAD/BAKERY
 - CELEBRATION
 - CLEANING
 - DAIRY
 - DELI
 - EGGS
 - FROZEN FOODS
 - GROCERY I
 - GROCERY II
 - HARDWARE
 - HOME AND KITCHEN I
 - HOME AND KITCHEN II
 - HOME APPLIANCES
 - HOME CARE
 - LADIESWEAR
 - LAWN AND GARDEN
 - LINGERIE
 - LIQUOR,WINE,BEER
 - MAGAZINES
 - MEATS
 - PERSONAL CARE
 - PET SUPPLIES
 - PLAYERS AND ELECTRONICS
 - POULTRY
 - PREPARED FOODS
 - PRODUCE
 - SCHOOL AND OFFICE SUPPLIES
 - SEAFOOD


### How to read it
The first block gives the date range, the number of calendar days in that range, the number of distinct dates actually present, and the difference. The second block lists every product family.

Results: sales run from 2013-01-01 to 2017-08-15, which is 1,688 calendar days. 1,684 distinct dates are present, so 4 days are missing. They are December 25 in 2013, 2014, 2015 and 2016. There are 33 product families.

### What it means
About four and a half years of daily history is plenty to learn weekly and yearly patterns. The four missing days are Christmas Day, when the stores were presumably closed, so they are not a data fault. Later we simply do not treat them as days of zero sales.

Of the 33 families, nine are perishable. We agreed on this shortlist: PRODUCE, DAIRY, MEATS, POULTRY, SEAFOOD, EGGS, BREAD/BAKERY, DELI and PREPARED FOODS. We left out FROZEN FOODS and GROCERY I (a large mixed category) for now. The first working version of the app will narrow this to a single family, chosen in Notebook 03.

### What's next and why
Step 4 looks for blanks and duplicates. Gaps or repeated rows would quietly distort totals and forecasts, so we check before aggregating anything.

## Step 4: Missing values and duplicates

### What we are looking for
Any column with blanks, and any repeated rows. Blanks and duplicates quietly distort totals and forecasts, so we find them now.

In [20]:
for name, df in tables.items():
    nulls = df.isna().sum()
    print(f"\n{name}: missing values per column")
    print(nulls[nulls > 0].to_string() if nulls.any() else "  none")
    print(f"  duplicate rows: {df.duplicated().sum():,}")


holidays_events: missing values per column
  none
  duplicate rows: 0

oil: missing values per column
dcoilwtico    43
  duplicate rows: 0

sample_submission: missing values per column
  none
  duplicate rows: 0

stores: missing values per column
  none
  duplicate rows: 0

test: missing values per column
  none
  duplicate rows: 0

train: missing values per column
  none
  duplicate rows: 0

transactions: missing values per column
  none
  duplicate rows: 0


### How to read it
For each file, the code lists only the columns that have blanks, with a count, or says "none". Then it shows the number of duplicate rows (rows that exactly repeat an earlier row).

Results: `train`, `stores`, `transactions` and `holidays_events` have no blanks and no duplicates. `oil` has no duplicates, but 43 of its 1,218 daily prices are blank (about 3.5%), including the very first row (2013-01-01). An extra check on the sales file found no negative sales and no family with zero sales for the whole period.

### What it means
The sales data, the part we depend on most, is clean, so there is nothing to fix before we aggregate it. The oil gap is small and matters only if we use oil price as a forecasting input; if we do, we fill each blank with the nearest earlier price, and the first row needs special handling. No negative sales means returns do not show up as negative demand.

One limit the audit cannot fix: the data records sales, not demand. A zero can mean nobody wanted the product or the shelf was empty, and we cannot tell which. The newsvendor model needs demand, so we keep this in mind and state it as an assumption in the write-up.

### What's next and why
With the audit passed, we write the Milestone 1 conclusion below and record it in the project plan. Then Milestone 2 builds the GIS foundation, because the stocking recommendation is per region.

## Conclusion for Milestone 1

**Result: the data passes the audit and we proceed with it.**

- **Files:** the Kaggle Store Sales download is complete. `train.csv` is the main file; the others are small lookup tables.
- **Coverage:** 2013-01-01 to 2017-08-15, 54 stores, 33 product families. The only missing dates are four Christmas Days.
- **Quality:** the sales file has no blanks, duplicates or negative values. The oil file has 43 blank prices out of 1,218.
- **Perishable shortlist (nine families):** PRODUCE, DAIRY, MEATS, POULTRY, SEAFOOD, EGGS, BREAD/BAKERY, DELI, PREPARED FOODS. The first app version narrows this to one family, chosen in Notebook 03.

**Limits to state openly in the write-up**
- Geography is only a city and state per store. We add coordinates, boundaries and warehouse sites ourselves, and label them as assumptions.
- There are no stock levels, costs, prices or shelf lives. Costs are labelled assumptions the user can change in the app.
- Sales are not demand: a zero may be a stockout.

**Work carried into later notebooks**
- Notebook 02 (GIS): find coordinates for the store cities and attach the boundaries.
- Notebook 03 (demand): choose the single family, check what share of its store-days have zero sales, and fill the oil gaps only if oil is used.

The result is recorded in the project plan's progress log.